# Full Segmentation Pipeline with Page-Level Metadata (Open-Source LLM)

**Document:** `pharma-blob-sample.pdf` (10 pages of bundled supplier documents)
**Model:** Mistral-7B-Instruct v0.2 (4-bit GGUF, run locally with llama.cpp). No Gemini, no API key.
**Author:** Bashar

For every page the pipeline outputs:

```json
{"page": 0, "is_new_doc": "Yes", "doc_type": "Cover Letter", "page_in_doc": 0}
```

**Design.** This builds on the approach that scored 10/10 in the boundary-detection task. The model is never asked to compare two pages. It describes each page on its own, and plain Python decides where documents start. Reliability comes from four layers:

1. **Constrained output:** llama.cpp forces the model's answer to be valid JSON in which `doc_type` can *only* be one of the fixed types.
2. **Verified identifiers:** an identifier the model returns is only used if it appears on the page; invented ones are discarded.
3. **Deterministic boundary rules in code:** continuation markers, type changes and identifier changes.
4. **Fallbacks for ambiguous pages:** blank pages, unclassifiable pages and failed calls, each flagged for human review.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## Step 0: Install dependencies

In [ ]:
%pip install -q llama-cpp-python==0.3.19 --only-binary=llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
%pip install -q pymupdf huggingface_hub

## Step 1: Model and PDF

In [1]:
import os, json, re, time
import llama_cpp
from llama_cpp import Llama

MODEL_PATH = os.environ.get("MISTRAL_GGUF", "/content/models/mistral-7b-instruct-v0.2.Q4_K_M.gguf")
if not os.path.exists(MODEL_PATH):
    from huggingface_hub import hf_hub_download
    hf_hub_download("TheBloke/Mistral-7B-Instruct-v0.2-GGUF", "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
                    local_dir=os.path.dirname(MODEL_PATH))

PDF_PATH = os.environ.get("BLOB_PDF", "/content/pharma-blob-sample.pdf")
if not os.path.exists(PDF_PATH):
    import urllib.request
    urllib.request.urlretrieve(
        "https://drive.google.com/uc?export=download&id=1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw", PDF_PATH)

N_GPU_LAYERS = -1 if llama_cpp.llama_supports_gpu_offload() else 0
llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=N_GPU_LAYERS, seed=42, verbose=False)
print(f"Mistral loaded on {'GPU' if N_GPU_LAYERS else 'CPU'} (chat format: {llm.chat_format})")

Mistral loaded on CPU (chat format: mistral-instruct)


## Step 2: Extract page text (PyMuPDF)

In [2]:
import fitz  # PyMuPDF

with fitz.open(PDF_PATH) as pdf:
    page_texts = [page.get_text() for page in pdf]
for i, text in enumerate(page_texts):
    first = [l.strip() for l in text.splitlines() if l.strip()][:1]
    print(f"page {i}: {len(text.split()):>3} words | {first[0] if first else '(blank)'}")

page 0: 191 words | Cytiva
page 1: 137 words | Certificate of Quality
page 2: 137 words | Certificate of Quality
page 3:  92 words | Packaging Component Specification
page 4: 101 words | Packaging Component Specification (continued)
page 5: 151 words | Declaration Regarding Transmissible Spongiform
page 6: 156 words | Material Description Sheet
page 7: 130 words | Supplier Qualification Record
page 8: 128 words | Supplier Qualification Record (continued)
page 9: 150 words | Cytiva


## Step 3: The page description call (constrained JSON)

`response_format` with a JSON schema makes llama.cpp restrict generation with a grammar. The model cannot produce invalid JSON, and `doc_type` can only be one of the eight allowed strings, so labels like "This Document Is A Material Description Or Product Information" or "Bse/Tse Declaration", which the course prompts produced earlier, are impossible.

The prompt gives a one-line definition of each type. Without them, Mistral called the cover letter a "Material Description" in testing.

In [3]:
DOC_TYPES = ["Cover Letter", "Certificate of Quality", "Packaging Specification", "BSE/TSE Declaration",
             "Material Description", "Supplier Qualification", "Chain of Custody", "Other"]

TYPE_DEFINITIONS = '''- Cover Letter: a letter addressed to the reader (greeting, sign-off), e.g. about storage or shipping
- Certificate of Quality: certifies one product lot (lot number, manufacture/expiry dates, release tests)
- Packaging Specification: packaging materials, configuration, packaging part numbers, drawings
- BSE/TSE Declaration: statement that products contain no animal-origin materials (BSE/TSE)
- Material Description: materials of construction and physical properties of a product
- Supplier Qualification: supplier approval status, audits, certifications, approved products
- Chain of Custody: where assemblies are manufactured and how they move to the customer
- Other: none of the above'''

PAGE_SCHEMA = {
    "type": "object",
    "properties": {"doc_type": {"type": "string", "enum": DOC_TYPES}, "identifier": {"type": "string"}},
    "required": ["doc_type", "identifier"],
}

from functools import lru_cache

@lru_cache(maxsize=None)  # same page text -> same answer at temperature 0, so re-runs reuse earlier calls
def describe_page(text):
    prompt = f'''Read this page from a bundle of pharmaceutical supplier documents.

Document types:
{TYPE_DEFINITIONS}

PAGE:
{text[:3000]}

Return JSON with:
"doc_type": the document type from the list above
"identifier": the document number, lot number or record code printed on this page that identifies this specific document. Copy it exactly as printed. Use "" if the page has none; do not make one up.'''
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object", "schema": PAGE_SCHEMA},
        temperature=0.0, max_tokens=80)
    return json.loads(response["choices"][0]["message"]["content"])

## Step 4: Verify identifiers, detect continuation markers

**Identifiers are never trusted as returned.** In testing, Mistral returned:

- `"18356721 (Lot Number for AKTA ready Low Flow Kit)"`, with extra words around the real lot number
- `"15102934, 28 9301 83, 20190712, 20210712, ..."`, several numbers at once
- `"29184612-AD"`, a part number and revision joined into a string that does not appear on the page
- `"Page 1 of 1"`, which is not an identifier at all
- for a signature-only page (Step 9), its approval date `"2023-11-08"`: real text on the page, but a date, not a document number
- with a shorter prompt, `"Cytiva-2022-06-03-AKTA-ready-Flow-Kit-Storage-Conditions"` for a page with no identifier: invented outright

So the code takes ID-like tokens (at least 5 characters, containing a digit, and not an ISO date) from the answer and keeps the first one that **literally appears on the page**. A hyphenated token that is not on the page is also tried piece by piece. If nothing matches, the identifier is empty. An invented identifier can therefore never cause a false split.

Continuation markers ("continued", "Page N of M" with N > 1) are detected with a regular expression rather than by the model.

In [4]:
ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def verified_identifier(model_identifier, page_text):
    page = re.sub(r"\s+", " ", page_text)
    for token in re.findall(r"[A-Za-z0-9][A-Za-z0-9-]*", model_identifier):
        if ISO_DATE.match(token):  # a date is not a document identifier
            continue
        for candidate in [token] + token.split("-"):
            if len(candidate) >= 5 and re.search(r"\d", candidate) and candidate in page:
                return candidate
    return ""

def has_continuation_marker(text):
    head = text[:400].lower()
    m = re.search(r"page\s+(\d+)\s*(?:of|/)\s*(\d+)", head)
    return "continued" in head or bool(m and int(m.group(1)) > 1)

# the cases seen in testing
for raw, page in [("18356721 (Lot Number for AKTA ready Low Flow Kit)", 1),
                  ("15102934, 28 9301 83, 20190712, 20210712, 15102934-AKTA-ready-High-Flow-Kit", 2),
                  ("29184612-AD", 6), ("Page 1 of 1", 9),
                  ("Cytiva-2022-06-03-AKTA-ready-Flow-Kit-Storage-Conditions", 0)]:
    print(f"page {page}: {raw!r:<80} -> {verified_identifier(raw, page_texts[page])!r}")

page 1: '18356721 (Lot Number for AKTA ready Low Flow Kit)'                              -> '18356721'
page 2: '15102934, 28 9301 83, 20190712, 20210712, 15102934-AKTA-ready-High-Flow-Kit'    -> '15102934'
page 6: '29184612-AD'                                                                    -> '29184612'
page 9: 'Page 1 of 1'                                                                    -> ''
page 0: 'Cytiva-2022-06-03-AKTA-ready-Flow-Kit-Storage-Conditions'                       -> ''


## Step 5: The segmentation loop, with fallbacks

| Situation | What happens | Flagged for review? |
|---|---|---|
| Page has (almost) no text, e.g. a blank or failed scan | No LLM call; treated as part of the current document | Yes |
| The LLM call fails | Retried once; if it fails again the page is treated as part of the current document | Yes |
| Page has a continuation marker | Same document, whatever the model said; if the model's type disagrees, the document's type is kept | Only if the types disagree |
| Model says "Other" and there is no evidence of a new document (no new identifier) | Treated as part of the current document; ambiguous pages such as a signature page are more likely to belong to the document before them than to start an unknown one | Yes |
| **Short page** (under 30 words) with no verified identifier, which the model says is a different type | Treated as part of the current document: too little evidence to start a new one. Added after the stress test in Step 9 showed a signature page being called a "Cover Letter" | Yes |
| Type changes, or both pages have verified identifiers and they differ | New document | No |
| Otherwise | Same document | No |

Every page of a document carries the type assigned to its first page, so a document can never change type halfway through.

In [5]:
MIN_CHARS = 20
SHORT_PAGE_WORDS = 30

def segment(texts, short_page_rule=True):
    rows, notes = [], []
    prev = None  # description of the previous page
    for i, text in enumerate(texts):
        flag = ""
        if len(text.strip()) < MIN_CHARS:
            info = {"doc_type": None, "identifier": "", "continuation": True}
            flag = "blank or unreadable page: kept with current document"
        else:
            for attempt in range(2):
                try:
                    raw = describe_page(text)
                    break
                except Exception as e:
                    raw = None
                    last_error = type(e).__name__
            if raw is None:
                info = {"doc_type": None, "identifier": "", "continuation": True}
                flag = f"LLM call failed twice ({last_error}): kept with current document"
            else:
                info = {"doc_type": raw["doc_type"],
                        "identifier": verified_identifier(raw["identifier"], text),
                        "model_identifier": raw["identifier"],
                        "continuation": has_continuation_marker(text)}

        if prev is None:  # first page always starts a document
            is_new, doc_type = True, info["doc_type"] or "Other"
        else:
            current_type = rows[-1]["doc_type"]
            if info["continuation"] or info["doc_type"] is None:
                is_new = False
                if info["doc_type"] not in (None, current_type) and not flag:
                    flag = f"continuation marker, but model said {info['doc_type']}: kept {current_type}"
            elif info["doc_type"] == "Other" and not (info["identifier"] and info["identifier"] != prev["identifier"]):
                is_new = False
                flag = "model could not classify the page: kept with current document"
            elif (short_page_rule and info["doc_type"] != current_type and not info["identifier"]
                  and len(text.split()) < SHORT_PAGE_WORDS):
                is_new = False
                flag = f"short page ({len(text.split())} words), model said {info['doc_type']}: kept with current document"
            else:
                type_changed = info["doc_type"] != current_type
                id_changed = bool(info["identifier"] and prev["identifier"] and info["identifier"] != prev["identifier"])
                is_new = type_changed or id_changed
            doc_type = info["doc_type"] if is_new else current_type

        page_in_doc = 0 if is_new else rows[-1]["page_in_doc"] + 1
        rows.append({"page": i, "is_new_doc": "Yes" if is_new else "No", "doc_type": doc_type, "page_in_doc": page_in_doc})
        notes.append({"page": i, "model_type": info["doc_type"], "model_identifier": info.get("model_identifier", ""),
                      "verified_identifier": info["identifier"], "continuation_marker": info["continuation"],
                      "needs_review": flag})
        if info["doc_type"] is not None:  # blank / failed pages don't replace the reference page
            prev = info
    return rows, notes

## Step 6: Run on the PDF

In [6]:
import pandas as pd

t0 = time.perf_counter()
results, notes = segment(page_texts)
print(f"{len(page_texts)} pages segmented in {time.perf_counter() - t0:.0f}s\n")
pd.DataFrame(notes).set_index("page")

10 pages segmented in 141s



,model_type,model_identifier,verified_identifier,continuation_marker,needs_review
page,,,,,
0,Cover Letter,,,False,
1,Certificate of Quality,18356721 (Lot Number for AKTA ready Low Flow Kit),18356721,False,
2,Certificate of Quality,"15102934, 28 9301 83, 20190712, 20210712, 1510...",15102934,False,
3,Packaging Specification,PKG-SPEC-2023-0847,PKG-SPEC-2023-0847,False,
4,Packaging Specification,PKG-SPEC-2023-0847,PKG-SPEC-2023-0847,True,
5,BSE/TSE Declaration,BSE-RA-2023-AKTA-001,BSE-RA-2023-AKTA-001,False,
6,Material Description,29184612-AD,29184612,False,
7,Supplier Qualification,SUP-2019-0847,SUP-2019-0847,False,
8,Supplier Qualification,SUP-2019-0847,SUP-2019-0847,True,


In [7]:
pd.DataFrame(results).set_index("page")

,is_new_doc,doc_type,page_in_doc
page,,,
0,Yes,Cover Letter,0
1,Yes,Certificate of Quality,0
2,Yes,Certificate of Quality,0
3,Yes,Packaging Specification,0
4,No,Packaging Specification,1
5,Yes,BSE/TSE Declaration,0
6,Yes,Material Description,0
7,Yes,Supplier Qualification,0
8,No,Supplier Qualification,1


## Step 7: Check against the correct answer

Worked out by reading every page: 8 documents. The Packaging Specification (pages 3–4) and the Supplier Qualification Record (pages 7–8) are two pages each, and pages 1 and 2 are two separate Certificates of Quality.

In [8]:
TRUTH = [
    {"page": 0, "is_new_doc": "Yes", "doc_type": "Cover Letter", "page_in_doc": 0},
    {"page": 1, "is_new_doc": "Yes", "doc_type": "Certificate of Quality", "page_in_doc": 0},
    {"page": 2, "is_new_doc": "Yes", "doc_type": "Certificate of Quality", "page_in_doc": 0},
    {"page": 3, "is_new_doc": "Yes", "doc_type": "Packaging Specification", "page_in_doc": 0},
    {"page": 4, "is_new_doc": "No", "doc_type": "Packaging Specification", "page_in_doc": 1},
    {"page": 5, "is_new_doc": "Yes", "doc_type": "BSE/TSE Declaration", "page_in_doc": 0},
    {"page": 6, "is_new_doc": "Yes", "doc_type": "Material Description", "page_in_doc": 0},
    {"page": 7, "is_new_doc": "Yes", "doc_type": "Supplier Qualification", "page_in_doc": 0},
    {"page": 8, "is_new_doc": "No", "doc_type": "Supplier Qualification", "page_in_doc": 1},
    {"page": 9, "is_new_doc": "Yes", "doc_type": "Chain of Custody", "page_in_doc": 0},
]

def check(results, truth):
    return {field: f"{sum(r[field] == t[field] for r, t in zip(results, truth))}/{len(truth)}"
            for field in ["is_new_doc", "doc_type", "page_in_doc"]} | {"all fields": results == truth}

check(results, TRUTH)

{'is_new_doc': '10/10',
 'doc_type': '10/10',
 'page_in_doc': '10/10',
 'all fields': True}

## Step 8: Final output

In [9]:
final_json = json.dumps(results, indent=2)
with open("segmentation_metadata.json", "w") as f:
    f.write(final_json)
print(final_json)

[
  {
    "page": 0,
    "is_new_doc": "Yes",
    "doc_type": "Cover Letter",
    "page_in_doc": 0
  },
  {
    "page": 1,
    "is_new_doc": "Yes",
    "doc_type": "Certificate of Quality",
    "page_in_doc": 0
  },
  {
    "page": 2,
    "is_new_doc": "Yes",
    "doc_type": "Certificate of Quality",
    "page_in_doc": 0
  },
  {
    "page": 3,
    "is_new_doc": "Yes",
    "doc_type": "Packaging Specification",
    "page_in_doc": 0
  },
  {
    "page": 4,
    "is_new_doc": "No",
    "doc_type": "Packaging Specification",
    "page_in_doc": 1
  },
  {
    "page": 5,
    "is_new_doc": "Yes",
    "doc_type": "BSE/TSE Declaration",
    "page_in_doc": 0
  },
  {
    "page": 6,
    "is_new_doc": "Yes",
    "doc_type": "Material Description",
    "page_in_doc": 0
  },
  {
    "page": 7,
    "is_new_doc": "Yes",
    "doc_type": "Supplier Qualification",
    "page_in_doc": 0
  },
  {
    "page": 8,
    "is_new_doc": "No",
    "doc_type": "Supplier Qualification",
    "page_in_doc": 1
  },
  {
 

## Step 9: Stress test of the fallbacks

The sample PDF is clean, so the fallbacks above were never triggered. To check that they work, the same pipeline is run on a modified page list with two common real-world problems inserted after page 4, the end of the packaging specification:

- **a blank page**, as produced by scanning the back of a single-sided sheet;
- **a signature-only page**: just an approval line, with no title or identifier.

Neither should start a new document: both belong to the packaging specification. Everything after them should still be segmented correctly.

In [10]:
signature_page = "Approved by: Packaging Engineering, Cytiva Uppsala\nSignature: ______________________\nDate: 2023-11-08\n"
stress_texts = page_texts[:5] + ["", signature_page] + page_texts[5:]

stress_truth = TRUTH[:5] + [
    {"page": 5, "is_new_doc": "No", "doc_type": "Packaging Specification", "page_in_doc": 2},
    {"page": 6, "is_new_doc": "No", "doc_type": "Packaging Specification", "page_in_doc": 3},
] + [{**t, "page": t["page"] + 2} for t in TRUTH[5:]]

def stress_table(results, notes):
    return pd.DataFrame([{**r, "model_type": n["model_type"], "model_identifier": n["model_identifier"],
                          "verified_identifier": n["verified_identifier"], "needs_review": n["needs_review"]}
                         for r, n in zip(results, notes)]).set_index("page")

# Without the short-page rule (the pipeline as first written)
before, before_notes = segment(stress_texts, short_page_rule=False)
print("Without short-page rule:", check(before, stress_truth))
stress_table(before, before_notes).loc[4:7]

Without short-page rule: {'is_new_doc': '11/12', 'doc_type': '11/12', 'page_in_doc': '11/12', 'all fields': False}


,is_new_doc,doc_type,page_in_doc,model_type,model_identifier,verified_identifier,needs_review
page,,,,,,,
4,No,Packaging Specification,1,Packaging Specification,PKG-SPEC-2023-0847,PKG-SPEC-2023-0847,
5,No,Packaging Specification,2,None,,,blank or unreadable page: kept with current do...
6,Yes,Cover Letter,0,Cover Letter,______________________2023-11-08,,
7,Yes,BSE/TSE Declaration,0,BSE/TSE Declaration,BSE-RA-2023-AKTA-001,BSE-RA-2023-AKTA-001,


In [11]:
# With the short-page rule (the pipeline above). Model answers are cached, so this makes no new LLM calls.
stress_results, stress_notes = segment(stress_texts)
print("With short-page rule:   ", check(stress_results, stress_truth))
print()
stress_table(stress_results, stress_notes)

With short-page rule:    {'is_new_doc': '12/12', 'doc_type': '12/12', 'page_in_doc': '12/12', 'all fields': True}



,is_new_doc,doc_type,page_in_doc,model_type,model_identifier,verified_identifier,needs_review
page,,,,,,,
0,Yes,Cover Letter,0,Cover Letter,,,
1,Yes,Certificate of Quality,0,Certificate of Quality,18356721 (Lot Number for AKTA ready Low Flow Kit),18356721,
2,Yes,Certificate of Quality,0,Certificate of Quality,"15102934, 28 9301 83, 20190712, 20210712, 1510...",15102934,
3,Yes,Packaging Specification,0,Packaging Specification,PKG-SPEC-2023-0847,PKG-SPEC-2023-0847,
4,No,Packaging Specification,1,Packaging Specification,PKG-SPEC-2023-0847,PKG-SPEC-2023-0847,
5,No,Packaging Specification,2,None,,,blank or unreadable page: kept with current do...
6,No,Packaging Specification,3,Cover Letter,______________________2023-11-08,,"short page (10 words), model said Cover Letter..."
7,Yes,BSE/TSE Declaration,0,BSE/TSE Declaration,BSE-RA-2023-AKTA-001,BSE-RA-2023-AKTA-001,
8,Yes,Material Description,0,Material Description,29184612-AD,29184612,


## Findings

Run with Mistral-7B on a laptop CPU: 10 pages in about 140 s, one LLM call per page. On a Colab T4 GPU it is much faster.

| Test | is_new_doc | doc_type | page_in_doc |
|---|---|---|---|
| Sample PDF (10 pages) | 10/10 | 10/10 | 10/10 |
| Stress test, before the short-page rule (12 pages) | 11/12 | 11/12 | 11/12 |
| Stress test, final pipeline (12 pages) | 12/12 | 12/12 | 12/12 |

**1. Constrained JSON removed a whole class of errors.** With the output grammar-constrained to the schema, every answer was valid JSON and every `doc_type` was one of the eight allowed labels. Free-text labels ("This Document Is A Material Description Or Product Information") and mangled ones ("Bse/Tse Declaration"), which broke the course prompts earlier, can no longer happen. The type definitions in the prompt still matter: without them, the cover letter came back as "Material Description".

**2. The model's identifiers cannot be trusted, but they can be checked.** Most identifiers came back padded ("18356721 (Lot Number for AKTA ready Low Flow Kit)"), combined (five numbers in one string), assembled ("29184612-AD" from the part number and revision), or wrong ("Page 1 of 1", a date). With a shorter prompt, one was invented outright. Keeping only an ID-like token that literally appears on the page turned all of these into either the right identifier or an empty one. An empty identifier is safe: it simply means the identifier does not influence the boundary decision.

**3. The edge cases were all about boundaries, not types.**
- Pages 1 and 2 are two Certificates of Quality with identical layouts; only the lot numbers (18356721 and 15102934) separate them.
- Pages 4 and 8 are second pages; the "continued" and "Page 2 of 2" markers keep them with the page before, and the matching document numbers agree.
- Pages 0 and 9 both start with the same Cytiva letterhead but are different document types.

**4. The stress test found a real failure, and fixing it took two changes.** A blank page was handled correctly by the blank-page fallback. A signature-only page was not: Mistral labelled it a Cover Letter and gave its date ("2023-11-08") as an identifier, so the pipeline started a new document there. The fixes:
- dates are never accepted as identifiers;
- a short page (under 30 words) with no verified identifier cannot start a new document, and is flagged for review instead.

Both rules were written *after* seeing this failure, so they are tuned to it. A short but genuine first page, such as a one-paragraph memo, would now be attached to the previous document. That is why such pages are flagged for review rather than silently merged.

**5. How reliability could be improved further.**
- Test on more and messier bundles, especially scanned pages that need OCR, where text extraction itself introduces errors.
- Send every flagged page to a person instead of trusting the fallback.
- Run a second model, or the same model with a differently worded prompt, and flag pages where the two disagree.
- Add more structural signals that do not need an LLM, such as page headers and footers, document-number patterns, and changes in layout.
